# Pandas: close the gap today

This is a **learning-by-doing** refresher, not a tour of methods. You will work with a small but realistic commerce dataset and repeatedly answer questions an analyst actually gets asked.

## How to use this notebook

1. Run the setup cells once.
2. For every exercise, **predict the shape and columns first**.
3. Write code from memory. Use the hint only after 3–5 honest minutes.
4. Check the companion solutions notebook only after you have a result.
5. When your answer differs, explain *why* in a new Markdown cell.

### Suggested one-day route

| Block | Sections | Time |
|---|---|---:|
| Warm-up | 0–2 | 75 min |
| Data wrangling | 3–4 | 90 min |
| Analysis | 5–6 | 90 min |
| Advanced patterns | 7–9 | 90 min |
| Capstone | 10 | 2–3 hr |

Exercises marked **Stretch** are optional today. The core track is enough to rebuild fluency.

> **Rule:** do not mutate the source tables (`customers`, `products`, `orders`, `order_items`, `returns`, `events`). Create a copy when an exercise needs mutation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:  # Keeps the notebook runnable in plain Python too.
    display = print

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("pandas", pd.__version__)

## 0. Build the practice dataset

Everything is deterministic. Re-running this cell restores the original tables.

**Data model**

- `customers`: one row per customer
- `products`: one row per product
- `orders`: one row per order
- `order_items`: one or more rows per order
- `returns`: a subset of returned order items
- `events`: customer browsing events

In [ ]:
rng = np.random.default_rng(42)

# Customers
n_customers = 60
customer_ids = np.arange(1001, 1001 + n_customers)
cities = np.array(["Riyadh", "Jeddah", "Dammam", "Makkah", "Medina", None], dtype=object)
segments = np.array(["Consumer", "Corporate", "Student"])
first_names = np.array(["Lina", "Omar", "Sara", "Noura", "Faisal", "Maya", "Ali", "Reem"])
last_names = np.array(["Khan", "Saleh", "Ahmed", "Noor", "Hassan", "Yousef"])

customers = pd.DataFrame({
    "customer_id": customer_ids,
    "first_name": rng.choice(first_names, n_customers),
    "last_name": rng.choice(last_names, n_customers),
    "segment": rng.choice(segments, n_customers, p=[0.55, 0.25, 0.20]),
    "city": rng.choice(cities, n_customers, p=[0.30, 0.25, 0.16, 0.12, 0.10, 0.07]),
    "signup_date": pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 730, n_customers), unit="D"),
    "birth_year": rng.choice([1985, 1990, 1995, 2000, 2003, np.nan], n_customers),
    "is_active": rng.choice([True, False], n_customers, p=[0.88, 0.12]),
})
customers["email"] = (
    " " + customers["first_name"].str.upper() + "." + customers["last_name"].str.lower()
    + customers["customer_id"].astype(str) + "@Example.COM "
)
customers["segment"] = customers["segment"].astype("category")

# Products
n_products = 24
product_ids = np.arange(201, 201 + n_products)
product_categories = np.array(["Electronics", "Home", "Office", "Fitness"])
category_for_product = np.repeat(product_categories, 6)
unit_cost = rng.integers(20, 450, n_products).astype(float)
markup = rng.uniform(1.25, 2.10, n_products)
products = pd.DataFrame({
    "product_id": product_ids,
    "product_name": [f"{cat[:3].upper()}-{i:02d}" for i, cat in enumerate(category_for_product, 1)],
    "category": category_for_product,
    "unit_cost": unit_cost,
    "list_price": np.round(unit_cost * markup, 2),
    "supplier": rng.choice(["Northstar", "Gulf Goods", "Vertex"], n_products),
})
products["category"] = products["category"].astype("category")

# Orders
n_orders = 420
order_ids = np.arange(50001, 50001 + n_orders)
order_ts = (
    pd.Timestamp("2025-01-01")
    + pd.to_timedelta(rng.integers(0, 181, n_orders), unit="D")
    + pd.to_timedelta(rng.integers(8, 23, n_orders), unit="h")
    + pd.to_timedelta(rng.integers(0, 60, n_orders), unit="m")
)
orders = pd.DataFrame({
    "order_id": order_ids,
    "customer_id": rng.choice(customer_ids, n_orders),
    "order_ts": order_ts,
    "channel": rng.choice(["Web", "App", "Store"], n_orders, p=[0.45, 0.35, 0.20]),
    "status": rng.choice(["completed", "cancelled", "pending"], n_orders, p=[0.86, 0.08, 0.06]),
    "shipping_fee": rng.choice([0.0, 15.0, 25.0], n_orders, p=[0.30, 0.48, 0.22]),
    "coupon_code": rng.choice([None, "WELCOME10", "SAVE20", "VIP"], n_orders, p=[0.65, 0.18, 0.12, 0.05]),
})

# Order items
item_rows = []
item_id = 1
product_lookup = products.set_index("product_id")["list_price"]
for order_id in order_ids:
    for product_id in rng.choice(product_ids, size=int(rng.integers(1, 6)), replace=False):
        list_price = product_lookup.loc[product_id]
        item_rows.append({
            "item_id": item_id,
            "order_id": order_id,
            "product_id": product_id,
            "quantity": int(rng.integers(1, 5)),
            "unit_price": round(float(list_price * rng.choice([0.95, 1.00, 1.05])), 2),
            "discount_pct": float(rng.choice([0, 0.05, 0.10, 0.20], p=[0.52, 0.20, 0.20, 0.08])),
        })
        item_id += 1
order_items = pd.DataFrame(item_rows)

# Returns (each item can appear at most once)
returned = order_items.sample(55, random_state=42)[["item_id", "order_id"]].copy()
returned["return_date"] = (
    returned["order_id"].map(orders.set_index("order_id")["order_ts"]).dt.normalize()
    + pd.to_timedelta(rng.integers(2, 31, len(returned)), unit="D")
)
returned["return_reason"] = rng.choice(
    ["Damaged", "Not as described", "Changed mind", "Wrong item"], len(returned)
)
returns = returned.reset_index(drop=True)

# Web/app events
n_events = 1000
events = pd.DataFrame({
    "event_id": np.arange(1, n_events + 1),
    "customer_id": rng.choice(customer_ids, n_events),
    "event_ts": (
        pd.Timestamp("2025-01-01")
        + pd.to_timedelta(rng.integers(0, 181, n_events), unit="D")
        + pd.to_timedelta(rng.integers(0, 24 * 60, n_events), unit="m")
    ),
    "event_type": rng.choice(["view", "search", "add_to_cart", "purchase"], n_events, p=[0.55, 0.20, 0.17, 0.08]),
    "device": rng.choice(["mobile", "desktop", "tablet"], n_events, p=[0.64, 0.29, 0.07]),
})

tables = {"customers": customers, "products": products, "orders": orders,
          "order_items": order_items, "returns": returns, "events": events}
pd.DataFrame({name: [df.shape[0], df.shape[1]] for name, df in tables.items()}, index=["rows", "columns"]).T

In [ ]:
def audit(df, name="DataFrame"):
    '''Compact inspection helper used throughout the notebook.'''
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(1),
        "unique": df.nunique(dropna=False),
    }).rename_axis(name)


def check_unique(df, keys):
    '''Raise early if keys that should be unique are not unique.'''
    if df.duplicated(keys).any():
        raise ValueError(f"Expected unique key(s): {keys}")
    return True


check_unique(customers, ["customer_id"])
check_unique(products, ["product_id"])
check_unique(orders, ["order_id"])
print("Setup complete — source tables are ready.")

## 1. Mental model: Series, DataFrame, index, dtype

A `Series` is a labeled one-dimensional array. A `DataFrame` is a collection of aligned Series. The index is a label system—not automatically a row number. Most pandas surprises come from **alignment**, **dtype**, or **grain** (what one row represents).

### Exercise 1: Construct and align Series

Create two Series with indexes `a, b, c` and `b, c, d`, add them, then add them again with missing values filled as zero. Observe label alignment.

**Target:** The first result has `NaN` at `a` and `d`; the second has no missing values.

<details><summary>Hint</summary>Use `Series.add(..., fill_value=0)` for the second result.</details>

**Worked solution:**

In [ ]:
s1 = pd.Series([10, 20, 30], index=["a", "b", "c"], name="left")
s2 = pd.Series([1, 2, 3], index=["b", "c", "d"], name="right")
aligned_sum = s1 + s2
filled_sum = s1.add(s2, fill_value=0)
aligned_sum, filled_sum

### Exercise 2: Create a tidy DataFrame

Create `mini` with columns `product`, `units`, and `price` from three equal-length Python lists. Add a vectorized `revenue` column.

<details><summary>Hint</summary>Pass a dict of lists to `pd.DataFrame`, then multiply two Series.</details>

**Worked solution:**

In [ ]:
mini = pd.DataFrame({
    "product": ["A", "B", "C"],
    "units": [3, 5, 2],
    "price": [10.0, 7.5, 20.0],
})
mini["revenue"] = mini["units"] * mini["price"]
mini

### Exercise 3: Audit before analysis

Inspect `customers` with `head`, `shape`, and the provided `audit()` helper. Then write one sentence in a Markdown cell: which columns need cleaning, and why?

<details><summary>Hint</summary>Try `display(customers.head())`, `customers.shape`, and `audit(customers)`.</details>

**Worked solution:**

In [ ]:
display(customers.head())
print(customers.shape)
display(audit(customers, "customers"))
# Likely cleaning targets: email (spaces/case), city and birth_year (missing values).

## 2. Selecting, filtering, sorting, and assigning

Keep these distinctions sharp:

- `df["col"]` → Series; `df[["col"]]` → DataFrame
- `.loc[row_labels, column_labels]` → labels / boolean masks
- `.iloc[row_positions, column_positions]` → integer positions
- use `&`, `|`, and `~` for elementwise conditions; parenthesize each condition
- use `.loc[mask, "column"] = value` for conditional assignment

### Exercise 4: Series versus one-column DataFrame

Select `email` from `customers` once as a Series and once as a DataFrame. Store their dimensions (`ndim`) in `selection_dims`.

**Target:** `(1, 2)`

**Worked solution:**

In [ ]:
email_series = customers["email"]
email_frame = customers[["email"]]
selection_dims = (email_series.ndim, email_frame.ndim)
selection_dims

### Exercise 5: Label versus position

Create `customer_view` indexed by `customer_id`. Select customer IDs 1005 through 1008 with `.loc`, then select the first four rows and first three columns with `.iloc`.

<details><summary>Hint</summary>Remember that label slices are inclusive; position slices exclude the stop.</details>

**Worked solution:**

In [ ]:
customer_view = customers.set_index("customer_id")
by_label = customer_view.loc[1005:1008]
by_position = customer_view.iloc[:4, :3]
display(by_label)
display(by_position)

### Exercise 6: Filter with multiple conditions

Find active Corporate customers in Riyadh or Jeddah. Keep only `customer_id`, `first_name`, `city`, and `segment`.

<details><summary>Hint</summary>Combine `.eq`, `.isin`, and `&`; wrap conditions in parentheses.</details>

**Worked solution:**

In [ ]:
mask = (
    customers["is_active"]
    & customers["segment"].eq("Corporate")
    & customers["city"].isin(["Riyadh", "Jeddah"])
)
target_customers = customers.loc[mask, ["customer_id", "first_name", "city", "segment"]]
target_customers.head()

### Exercise 7: Filter with query

Use `query` to select Web orders that are not cancelled and have a shipping fee of at least 15. Sort newest first.

<details><summary>Hint</summary>Inside `query`, column names can be referenced directly.</details>

**Worked solution:**

In [ ]:
web_orders = (
    orders.query("channel == 'Web' and status != 'cancelled' and shipping_fee >= 15")
          .sort_values("order_ts", ascending=False)
)
web_orders.head()

### Exercise 8: Safe conditional assignment

On a copy named `orders_labeled`, create `shipping_tier`: `free` when the fee is zero, `standard` at 15, and `express` at 25. Use `.loc` assignments.

<details><summary>Hint</summary>Initialize the column, then assign each condition with `df.loc[mask, col] = value`.</details>

**Worked solution:**

In [ ]:
orders_labeled = orders.copy()
orders_labeled["shipping_tier"] = pd.NA
orders_labeled.loc[orders_labeled["shipping_fee"].eq(0), "shipping_tier"] = "free"
orders_labeled.loc[orders_labeled["shipping_fee"].eq(15), "shipping_tier"] = "standard"
orders_labeled.loc[orders_labeled["shipping_fee"].eq(25), "shipping_tier"] = "express"
orders_labeled["shipping_tier"].value_counts(dropna=False)

### Exercise 9: Top-N without sorting everything

Return the five products with the largest absolute markup (`list_price - unit_cost`). Include a `markup` column.

<details><summary>Hint</summary>Use `.assign()` followed by `.nlargest()`.</details>

**Worked solution:**

In [ ]:
product_margin = products.assign(markup=products["list_price"] - products["unit_cost"])
top_markup = product_margin.nlargest(5, "markup")
top_markup[["product_id", "product_name", "category", "markup"]]

## 3. Cleaning: missing values, strings, types, dates, duplicates

Cleaning is not a ritual. For each column, decide:

1. What does missing mean?
2. What dtype matches the domain?
3. What values are valid?
4. What constitutes a duplicate at this table's grain?

### Exercise 10: Normalize text

On a copy of `customers`, strip and lowercase `email`. Create `full_name` from first and last names with one space between them.

<details><summary>Hint</summary>Chain `.str.strip().str.lower()`; use `.str.cat(..., sep=' ')`.</details>

**Worked solution:**

In [ ]:
customers_clean = customers.copy()
customers_clean["email"] = customers_clean["email"].str.strip().str.lower()
customers_clean["full_name"] = customers_clean["first_name"].str.cat(customers_clean["last_name"], sep=" ")
customers_clean[["email", "full_name"]].head()

### Exercise 11: Handle missing values intentionally

On `customers_clean`, fill missing city with `Unknown`. Fill missing birth year with the median birth year *within each segment*, then convert it to nullable integer `Int64`.

<details><summary>Hint</summary>`groupby(...)[col].transform('median')` returns a Series aligned to the original rows.</details>

**Worked solution:**

In [ ]:
customers_clean["city"] = customers_clean["city"].fillna("Unknown")
segment_median = customers_clean.groupby("segment", observed=True)["birth_year"].transform("median")
customers_clean["birth_year"] = customers_clean["birth_year"].fillna(segment_median).round().astype("Int64")
audit(customers_clean.loc[:, ["city", "birth_year"]])

### Exercise 12: Parse messy values safely

Create the messy Series below. Convert it to numbers so invalid values become missing, then compute its mean.

**Target:** Mean ≈ 23.50

<details><summary>Hint</summary>Use `pd.to_numeric(..., errors='coerce')`.</details>

**Worked solution:**

In [ ]:
messy_amounts = pd.Series(["10.5", " 20 ", "N/A", "bad", None, "40"])
clean_amounts = pd.to_numeric(messy_amounts, errors="coerce")
clean_mean = clean_amounts.mean()
clean_amounts, clean_mean

### Exercise 13: Engineer date features

From `orders.order_ts`, create `order_date`, monthly period `order_month`, weekday name, and `is_weekend` on a copy named `orders_enriched`.

<details><summary>Hint</summary>Use the `.dt` accessor: `normalize`, `to_period`, `day_name`, `dayofweek`.</details>

**Worked solution:**

In [ ]:
orders_enriched = orders.copy()
orders_enriched = orders_enriched.assign(
    order_date=orders_enriched["order_ts"].dt.normalize(),
    order_month=orders_enriched["order_ts"].dt.to_period("M"),
    weekday=orders_enriched["order_ts"].dt.day_name(),
    is_weekend=orders_enriched["order_ts"].dt.dayofweek.ge(5),
)
orders_enriched.head()

### Exercise 14: Find duplicates at the correct grain

Verify that `item_id` is unique. Then count how many order-product combinations repeat. Explain why repeated `order_id` values alone are expected in `order_items`.

<details><summary>Hint</summary>Use `.duplicated(subset).sum()` with the candidate key(s).</details>

**Worked solution:**

In [ ]:
item_id_duplicates = order_items.duplicated("item_id").sum()
order_product_duplicates = order_items.duplicated(["order_id", "product_id"]).sum()
item_id_duplicates, order_product_duplicates
# Repeated order_id is expected because one order contains multiple line items.

### Exercise 15: Categoricals and memory

Compare the deep memory use of `orders.channel` before and after conversion to category. Store the two byte counts in `channel_memory`.

<details><summary>Hint</summary>Use `.memory_usage(deep=True)`.</details>

**Worked solution:**

In [ ]:
channel_memory = {
    "object_bytes": orders["channel"].memory_usage(deep=True),
    "category_bytes": orders["channel"].astype("category").memory_usage(deep=True),
}
channel_memory

## 4. Feature engineering and method chaining

Prefer vectorized expressions and readable chains. Use `.assign()` for new columns, `.pipe()` for named transformations, and break long chains across lines. `apply(axis=1)` is often a slow Python loop in disguise.

### Exercise 16: Build the analysis-ready line-item table

Merge `order_items` with selected order and product columns. Keep only completed orders. Add `gross_sales`, `discount_amount`, `net_sales`, `cost`, and `profit`. Validate each merge as many-to-one.

<details><summary>Hint</summary>Merge order_items→orders and result→products with `validate='many_to_one'`. In `.assign`, later lambdas can use columns created earlier.</details>

**Worked solution:**

In [ ]:
line_items = (
    order_items
    .merge(
        orders[["order_id", "customer_id", "order_ts", "channel", "status"]],
        on="order_id", how="left", validate="many_to_one"
    )
    .merge(
        products[["product_id", "product_name", "category", "unit_cost"]],
        on="product_id", how="left", validate="many_to_one"
    )
    .loc[lambda d: d["status"].eq("completed")]
    .assign(
        gross_sales=lambda d: d["quantity"] * d["unit_price"],
        discount_amount=lambda d: d["quantity"] * d["unit_price"] * d["discount_pct"],
        net_sales=lambda d: d["gross_sales"] - d["discount_amount"],
        cost=lambda d: d["quantity"] * d["unit_cost"],
        profit=lambda d: d["net_sales"] - d["cost"],
    )
)
line_items.head()

### Exercise 17: Replace row-wise apply with vectorization

Create `discount_band` on a copy of `line_items`: `none` for 0, `light` through 5%, `medium` through 10%, and `heavy` above 10%. Use `pd.cut`, not `apply(axis=1)`.

<details><summary>Hint</summary>Provide ordered bin edges and four labels to `pd.cut`.</details>

**Worked solution:**

In [ ]:
items_banded = line_items.copy()
items_banded["discount_band"] = pd.cut(
    items_banded["discount_pct"],
    bins=[-0.001, 0, 0.05, 0.10, 1],
    labels=["none", "light", "medium", "heavy"],
)
items_banded["discount_band"].value_counts(sort=False)

### Exercise 18: Reusable pipe function

Write `add_margin_metrics(df)` that returns the input with `margin_pct = profit / net_sales` and `is_loss = profit < 0`. Apply it to `line_items` using `.pipe()`.

<details><summary>Hint</summary>Return `df.assign(...)`, then call `line_items.pipe(your_function)`.</details>

**Worked solution:**

In [ ]:
def add_margin_metrics(df):
    return df.assign(
        margin_pct=lambda d: d["profit"].div(d["net_sales"]).where(d["net_sales"].ne(0)),
        is_loss=lambda d: d["profit"].lt(0),
    )

items_with_margin = line_items.pipe(add_margin_metrics)
items_with_margin[["net_sales", "profit", "margin_pct", "is_loss"]].head()

## 5. GroupBy: split, apply, combine

Before grouping, say the output grain aloud: “one row per ___.”

- `.agg()` reduces groups
- `.transform()` returns one value per original row
- `.filter()` keeps or removes whole groups
- named aggregation gives clean output column names

### Exercise 19: Named aggregation

Create one row per category with total revenue, total profit, units sold, order count, and average discount. Sort by revenue descending.

<details><summary>Hint</summary>Use `.groupby(...).agg(new_name=(source_column, 'function'))`.</details>

**Worked solution:**

In [ ]:
category_kpis = (
    line_items.groupby("category", observed=True)
    .agg(
        revenue=("net_sales", "sum"),
        profit=("profit", "sum"),
        units=("quantity", "sum"),
        orders=("order_id", "nunique"),
        avg_discount=("discount_pct", "mean"),
    )
    .sort_values("revenue", ascending=False)
    .reset_index()
)
category_kpis

### Exercise 20: Transform: share within group

For every line item, calculate its share of the order's net sales. Confirm that shares sum to 1 per order (within floating-point tolerance).

<details><summary>Hint</summary>`groupby('order_id')['net_sales'].transform('sum')` aligns totals back to rows.</details>

**Worked solution:**

In [ ]:
items_share = line_items.copy()
items_share["order_revenue"] = items_share.groupby("order_id")["net_sales"].transform("sum")
items_share["share_of_order"] = items_share["net_sales"] / items_share["order_revenue"]
share_check = items_share.groupby("order_id")["share_of_order"].sum()
share_check.head()
# Optional: np.allclose(share_check, 1.0)

### Exercise 21: Multi-key grouping

Create monthly revenue by channel. Use a monthly period derived from `order_ts`. Return a tidy DataFrame sorted by month and revenue descending within month.

<details><summary>Hint</summary>Create month before grouping, then group on `['month', 'channel']`.</details>

**Worked solution:**

In [ ]:
monthly_channel = (
    line_items.assign(month=lambda d: d["order_ts"].dt.to_period("M"))
    .groupby(["month", "channel"], as_index=False)
    .agg(revenue=("net_sales", "sum"))
    .sort_values(["month", "revenue"], ascending=[True, False])
)
monthly_channel.head(10)

### Exercise 22: Filter whole groups

Keep all rows belonging to customers whose completed lifetime revenue exceeds 10,000. How many customers and rows remain?

<details><summary>Hint</summary>Use `groupby(...).filter(lambda group: condition)`.</details>

**Worked solution:**

In [ ]:
high_value_rows = line_items.groupby("customer_id").filter(lambda g: g["net_sales"].sum() > 10_000)
result = {
    "customers": high_value_rows["customer_id"].nunique(),
    "rows": len(high_value_rows),
}
result

### Exercise 23: Frequency tables

Build a normalized cross-tab of order `channel` by `status`, where each row sums to 1.

<details><summary>Hint</summary>Use `pd.crosstab(..., normalize='index')`.</details>

**Worked solution:**

In [ ]:
channel_status = pd.crosstab(orders["channel"], orders["status"], normalize="index")
channel_status

## 6. Combining tables: concat, merge, join

Think in database terms:

- identify the key(s)
- state the relationship: one-to-one, one-to-many, or many-to-many
- choose inner/left/right/outer intentionally
- validate cardinality with `validate=`
- use `indicator=True` to audit matches
- check row counts before and after

### Exercise 24: Merge and preserve grain

Attach customer segment and city to `orders` with a left merge. Validate many-to-one. Verify that row count and unique order count did not change.

<details><summary>Hint</summary>`orders` has many rows per customer; `customers` has one row per customer.</details>

**Worked solution:**

In [ ]:
orders_customers = orders.merge(
    customers[["customer_id", "segment", "city"]],
    on="customer_id", how="left", validate="many_to_one"
)
validation = {
    "same_rows": len(orders_customers) == len(orders),
    "same_orders": orders_customers["order_id"].nunique() == orders["order_id"].nunique(),
}
validation

### Exercise 25: Anti-join

Find customers who placed no orders. Use a left merge with `indicator=True`; return their IDs and names.

<details><summary>Hint</summary>Merge against distinct ordered customer IDs, then filter `_merge == 'left_only'`.</details>

**Worked solution:**

In [ ]:
customer_order_match = customers.merge(
    orders[["customer_id"]].drop_duplicates(),
    on="customer_id", how="left", indicator=True, validate="one_to_one"
)
no_orders = customer_order_match.loc[
    customer_order_match["_merge"].eq("left_only"),
    ["customer_id", "first_name", "last_name"]
]
no_orders

### Exercise 26: Return flag without inflating rows

Add `was_returned` to `order_items` using `item_id`. The result must have exactly the same row count as `order_items`.

<details><summary>Hint</summary>Because `item_id` is unique in both tables, validate `one_to_one`.</details>

**Worked solution:**

In [ ]:
items_returns = order_items.merge(
    returns[["item_id", "return_reason"]],
    on="item_id", how="left", validate="one_to_one"
)
items_returns["was_returned"] = items_returns["return_reason"].notna()
len(items_returns), items_returns["was_returned"].sum()

### Exercise 27: Concatenate partitions

Split `orders` into Jan–Mar and Apr–Jun partitions, then concatenate them back with `ignore_index=True`. Verify equality after sorting by `order_id`.

<details><summary>Hint</summary>Use `pd.concat([part1, part2], ignore_index=True)` and `.equals()`.</details>

**Worked solution:**

In [ ]:
first_half = orders.loc[orders["order_ts"].dt.month.le(3)]
second_half = orders.loc[orders["order_ts"].dt.month.gt(3)]
recombined = pd.concat([first_half, second_half], ignore_index=True)
are_equal = (
    recombined.sort_values("order_id").reset_index(drop=True)
    .equals(orders.sort_values("order_id").reset_index(drop=True))
)
are_equal

### Exercise 28: Nearest-time join with merge_asof · 🧠 Stretch

For every purchase event, attach the most recent earlier order by the same customer. Calculate hours since that order. Inputs must be sorted correctly.

<details><summary>Hint</summary>Sort primarily on the time key, then call `pd.merge_asof(..., by='customer_id', direction='backward')`.</details>

**Worked solution:**

In [ ]:
purchases = (
    events.loc[events["event_type"].eq("purchase"), ["event_id", "customer_id", "event_ts"]]
    .sort_values(["event_ts", "customer_id"])
)
order_times = orders[["order_id", "customer_id", "order_ts"]].sort_values(["order_ts", "customer_id"])
purchase_context = pd.merge_asof(
    purchases, order_times,
    left_on="event_ts", right_on="order_ts", by="customer_id",
    direction="backward"
)
purchase_context["hours_since_order"] = (
    purchase_context["event_ts"] - purchase_context["order_ts"]
).dt.total_seconds().div(3600)
purchase_context.head()

## 7. Reshaping: wide, long, pivot, melt, explode

Tidy data usually means: one variable per column, one observation per row, one type of observational unit per table. Reshaping changes representation, not underlying facts.

### Exercise 29: Pivot with aggregation

Create a table with month as rows, category as columns, and summed net sales as values. Fill missing cells with zero and add row/column totals.

<details><summary>Hint</summary>Use `pd.pivot_table` with `aggfunc='sum'`, `fill_value=0`, and `margins=True`.</details>

**Worked solution:**

In [ ]:
month_category_pivot = pd.pivot_table(
    line_items.assign(month=line_items["order_ts"].dt.to_period("M")),
    index="month", columns="category", values="net_sales",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total", observed=True
)
month_category_pivot

### Exercise 30: Melt wide data to long

Reset the pivot index, remove the Total row, and melt category columns into `category` and `revenue`. Exclude the Total column.

<details><summary>Hint</summary>Clean the margins first, then `.melt(id_vars='month', ...)`.</details>

**Worked solution:**

In [ ]:
wide_no_totals = month_category_pivot.drop(index="Total").drop(columns="Total").reset_index()
long_revenue = wide_no_totals.melt(
    id_vars="month", var_name="category", value_name="revenue"
)
long_revenue.head()

### Exercise 31: Unstack a grouped Series

Count orders by status and channel using `groupby().size()`, then unstack channel into columns and fill missing values with zero.

<details><summary>Hint</summary>`groupby([...]).size()` produces a MultiIndex Series; `.unstack()` moves one level to columns.</details>

**Worked solution:**

In [ ]:
status_channel_counts = (
    orders.groupby(["status", "channel"]).size().unstack("channel", fill_value=0)
)
status_channel_counts

### Exercise 32: Explode list-like values

Create the small `campaigns` DataFrame below, explode `channels` so each channel gets one row, then count campaigns per channel.

<details><summary>Hint</summary>Use `.explode('channels', ignore_index=True)`.</details>

**Worked solution:**

In [ ]:
campaigns = pd.DataFrame({
    "campaign": ["Launch", "Ramadan", "BackToSchool"],
    "channels": [["Web", "App"], ["Web", "Store", "App"], ["Store"]],
})
campaign_channels = campaigns.explode("channels", ignore_index=True)
channel_counts = campaign_channels["channels"].value_counts()
channel_counts

## 8. Time series, windows, ranking

Window operations preserve row-level context while looking backward or within groups:

- `shift` → previous/next row
- `diff` / `pct_change` → change from prior row
- `rolling` → moving window
- `expanding` → all observations so far
- `rank` → relative position

### Exercise 33: Daily resampling and rolling average

Create daily completed revenue indexed by date, including zero-revenue dates, then calculate a 7-day rolling mean with at least one observation.

<details><summary>Hint</summary>Set a DatetimeIndex, `.resample('D').sum()`, then `.rolling(7, min_periods=1).mean()`.</details>

**Worked solution:**

In [ ]:
daily_revenue = (
    line_items.set_index("order_ts")["net_sales"]
    .resample("D").sum()
)
rolling_7d = daily_revenue.rolling(7, min_periods=1).mean()
pd.DataFrame({"revenue": daily_revenue, "rolling_7d": rolling_7d}).head(10)

### Exercise 34: Previous order with shift

At one row per completed order, calculate each customer's previous order timestamp and days since previous order.

<details><summary>Hint</summary>Sort within customer before `groupby('customer_id')['order_ts'].shift()`.</details>

**Worked solution:**

In [ ]:
order_level = (
    line_items.groupby(["order_id", "customer_id", "order_ts"], as_index=False)
    .agg(order_revenue=("net_sales", "sum"))
    .sort_values(["customer_id", "order_ts"])
)
order_level["previous_order_ts"] = order_level.groupby("customer_id")["order_ts"].shift()
order_level["days_since_previous"] = (
    order_level["order_ts"] - order_level["previous_order_ts"]
).dt.total_seconds().div(86_400)
order_level.head(10)

### Exercise 35: Cumulative customer value

On `order_level`, create each customer's cumulative revenue and sequential order number.

<details><summary>Hint</summary>Use grouped `.cumsum()` and `groupby(...).cumcount() + 1`.</details>

**Worked solution:**

In [ ]:
order_level["cumulative_revenue"] = order_level.groupby("customer_id")["order_revenue"].cumsum()
order_level["order_number"] = order_level.groupby("customer_id").cumcount().add(1)
order_level.head(10)

### Exercise 36: Rank products within category

Aggregate product revenue, then rank products by revenue within category. Keep the top 3 per category. Break ties with first appearance order.

<details><summary>Hint</summary>Aggregate first, then `groupby('category')['revenue'].rank(method='first', ascending=False)`.</details>

**Worked solution:**

In [ ]:
product_revenue = (
    line_items.groupby(["category", "product_id", "product_name"], observed=True, as_index=False)
    .agg(revenue=("net_sales", "sum"))
    .sort_values(["category", "revenue"], ascending=[True, False])
)
product_revenue["category_rank"] = product_revenue.groupby("category", observed=True)["revenue"].rank(
    method="first", ascending=False
)
top3_per_category = product_revenue.loc[product_revenue["category_rank"].le(3)]
top3_per_category

### Exercise 37: Customer value quartiles

Aggregate completed revenue per customer and assign quartiles Q1–Q4 with `qcut`, where Q4 is highest value.

<details><summary>Hint</summary>Use `pd.qcut(series, q=4, labels=[...])`.</details>

**Worked solution:**

In [ ]:
customer_value = (
    line_items.groupby("customer_id", as_index=False)
    .agg(lifetime_revenue=("net_sales", "sum"))
)
customer_value["value_quartile"] = pd.qcut(
    customer_value["lifetime_revenue"], q=4, labels=["Q1", "Q2", "Q3", "Q4"]
)
customer_value.sort_values("lifetime_revenue", ascending=False).head()

## 9. Debugging and performance habits

The fastest pandas users are not the ones who remember the most methods. They detect wrong grain, silent row multiplication, dtype drift, and accidental mutation early.

### Exercise 38: Fix chained assignment

The intended result is a copy of orders where high shipping fees are labeled `expensive`. Rewrite the broken pattern safely: `df[df['shipping_fee'] > 15]['shipping_label'] = 'expensive'`.

<details><summary>Hint</summary>Assign with `.loc[mask, 'shipping_label']` on the original copy.</details>

**Worked solution:**

In [ ]:
orders_debug = orders.copy()
orders_debug["shipping_label"] = "regular"
orders_debug.loc[orders_debug["shipping_fee"].gt(15), "shipping_label"] = "expensive"
orders_debug["shipping_label"].value_counts(dropna=False)

### Exercise 39: Diagnose a many-to-many merge

Create `customer_categories` as distinct customer-category pairs from `line_items`. Predict why merging it to `orders` only on `customer_id` increases rows. Perform the merge and quantify the inflation factor.

<details><summary>Hint</summary>The right table is not unique on `customer_id`; inspect `.duplicated('customer_id')`.</details>

**Worked solution:**

In [ ]:
customer_categories = line_items[["customer_id", "category"]].drop_duplicates()
orders_expanded = orders.merge(
    customer_categories, on="customer_id", how="left", validate="many_to_many"
)
inflation_factor = len(orders_expanded) / len(orders)
inflation_factor
# Each customer can have several categories, so each order repeats once per customer-category.

### Exercise 40: Use eval for a vectorized expression · 🧠 Stretch

Recreate gross sales and discounted net sales from `order_items` with `.eval()`. Compare the result to an ordinary vectorized expression using `np.allclose`.

<details><summary>Hint</summary>Pass a multiline expression string to `.eval()`.</details>

**Worked solution:**

In [ ]:
evaluated = order_items.copy().eval(
    "gross = quantity * unit_price\nnet = gross * (1 - discount_pct)"
)
same_result = np.allclose(
    evaluated["net"],
    order_items["quantity"] * order_items["unit_price"] * (1 - order_items["discount_pct"])
)
same_result

### Exercise 41: Round-trip a typed file · 🧠 Stretch

Write `line_items` to a temporary Parquet file and read it back. Confirm shape and dtypes. This avoids polluting your workspace.

<details><summary>Hint</summary>Use `.to_parquet(path, index=False)` and `pd.read_parquet(path)`.</details>

**Worked solution:**

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "line_items.parquet"
    line_items.to_parquet(path, index=False)
    restored = pd.read_parquet(path)

roundtrip_ok = {
    "same_shape": restored.shape == line_items.shape,
    "same_dtypes": restored.dtypes.equals(line_items.dtypes),
}
roundtrip_ok

## 10. Capstone: six-month commerce review

You are the analyst for this retailer. Leadership wants a compact performance review using completed orders only. Build each table from the source data, even if a similar intermediate exists above.

**Quality bar**

- every table has a stated grain
- merges use `validate=`
- financial columns are explicit
- missing values are handled intentionally
- tables are sorted for human reading
- your conclusions cite numbers

### Exercise 42: Capstone A — analysis base

Build `capstone_items`: one row per completed order item, with customer segment/city, product category/cost, and all financial columns. Add a boolean return flag without changing the grain.

<details><summary>Hint</summary>Merge items→orders→products→customers→returns. The relationship is many-to-one except returns on unique item ID.</details>

**Worked solution:**

In [ ]:
capstone_items = (
    order_items
    .merge(orders, on="order_id", how="left", validate="many_to_one")
    .merge(products, on="product_id", how="left", validate="many_to_one")
    .merge(customers[["customer_id", "segment", "city"]], on="customer_id", how="left", validate="many_to_one")
    .merge(returns[["item_id", "return_reason"]], on="item_id", how="left", validate="one_to_one")
    .loc[lambda d: d["status"].eq("completed")]
    .assign(
        gross_sales=lambda d: d["quantity"] * d["unit_price"],
        discount_amount=lambda d: d["gross_sales"] * d["discount_pct"],
        net_sales=lambda d: d["gross_sales"] - d["discount_amount"],
        cost=lambda d: d["quantity"] * d["unit_cost"],
        profit=lambda d: d["net_sales"] - d["cost"],
        was_returned=lambda d: d["return_reason"].notna(),
    )
)
assert capstone_items["item_id"].is_unique
assert capstone_items["order_id"].nunique() == orders.loc[orders["status"].eq("completed"), "order_id"].nunique()
capstone_items.head()

### Exercise 43: Capstone B — executive KPI card

Produce a one-row DataFrame with revenue, profit, margin %, completed orders, customers, units, average order value, and item return rate.

<details><summary>Hint</summary>AOV uses unique orders, not line-item rows. Return rate here is returned items / items.</details>

**Worked solution:**

In [ ]:
total_revenue = capstone_items["net_sales"].sum()
total_profit = capstone_items["profit"].sum()
completed_orders = capstone_items["order_id"].nunique()
executive_kpis = pd.DataFrame({
    "revenue": [total_revenue],
    "profit": [total_profit],
    "margin_pct": [total_profit / total_revenue],
    "completed_orders": [completed_orders],
    "customers": [capstone_items["customer_id"].nunique()],
    "units": [capstone_items["quantity"].sum()],
    "avg_order_value": [total_revenue / completed_orders],
    "item_return_rate": [capstone_items["was_returned"].mean()],
})
executive_kpis

### Exercise 44: Capstone C — monthly trend

Create monthly revenue, profit, orders, AOV, and month-over-month revenue growth. Plot revenue and profit on one chart.

<details><summary>Hint</summary>Group by a monthly Period, then calculate AOV and `.pct_change()` after sorting.</details>

**Worked solution:**

In [ ]:
monthly_kpis = (
    capstone_items.assign(month=lambda d: d["order_ts"].dt.to_period("M"))
    .groupby("month", as_index=False)
    .agg(
        revenue=("net_sales", "sum"),
        profit=("profit", "sum"),
        orders=("order_id", "nunique"),
    )
    .sort_values("month")
)
monthly_kpis["aov"] = monthly_kpis["revenue"] / monthly_kpis["orders"]
monthly_kpis["revenue_growth"] = monthly_kpis["revenue"].pct_change()
monthly_kpis.assign(month=monthly_kpis["month"].astype(str)).plot(
    x="month", y=["revenue", "profit"], marker="o", figsize=(9, 4), title="Monthly performance"
)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
monthly_kpis

### Exercise 45: Capstone D — category scorecard

Create one row per category with revenue, profit, margin %, return rate, units, and revenue share. Rank categories by revenue.

<details><summary>Hint</summary>Aggregate first; derive ratios afterward to avoid averaging row-level ratios.</details>

**Worked solution:**

In [ ]:
category_scorecard = (
    capstone_items.groupby("category", observed=True, as_index=False)
    .agg(
        revenue=("net_sales", "sum"),
        profit=("profit", "sum"),
        return_rate=("was_returned", "mean"),
        units=("quantity", "sum"),
    )
    .assign(
        margin_pct=lambda d: d["profit"] / d["revenue"],
        revenue_share=lambda d: d["revenue"] / d["revenue"].sum(),
    )
    .sort_values("revenue", ascending=False)
)
category_scorecard["revenue_rank"] = category_scorecard["revenue"].rank(method="dense", ascending=False).astype(int)
category_scorecard

### Exercise 46: Capstone E — RFM segmentation · 🧠 Stretch

At customer grain, compute recency in days from 2025-07-01, completed-order frequency, and monetary value. Score each metric 1–4 with quantiles and create an RFM string such as `444`.

<details><summary>Hint</summary>Aggregate last order, unique order count, and revenue. Rank before `qcut` to avoid duplicate bin-edge problems.</details>

**Worked solution:**

In [ ]:
snapshot_date = pd.Timestamp("2025-07-01")
rfm = (
    capstone_items.groupby("customer_id", as_index=False)
    .agg(
        last_order=("order_ts", "max"),
        frequency=("order_id", "nunique"),
        monetary=("net_sales", "sum"),
    )
)
rfm["recency"] = (snapshot_date - rfm["last_order"].dt.normalize()).dt.days
rfm["r_score"] = pd.qcut(rfm["recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["rfm_segment"] = rfm[["r_score", "f_score", "m_score"]].astype(str).agg("".join, axis=1)
rfm.sort_values(["r_score", "f_score", "m_score"], ascending=False).head(10)

### Exercise 47: Capstone F — cohort retention · 🧠 Stretch

Define each customer's acquisition month as their first completed order month. Build a cohort table of unique active customers by months since acquisition, then divide by cohort size to get retention.

<details><summary>Hint</summary>Reduce to unique customer-months. Convert monthly Periods to integer ordinals to subtract them.</details>

**Worked solution:**

In [ ]:
customer_months = (
    capstone_items.assign(activity_month=lambda d: d["order_ts"].dt.to_period("M"))
    [["customer_id", "activity_month"]]
    .drop_duplicates()
)
customer_months["acquisition_month"] = customer_months.groupby("customer_id")["activity_month"].transform("min")
customer_months["period_number"] = (
    customer_months["activity_month"].astype("int64")
    - customer_months["acquisition_month"].astype("int64")
)
cohort_counts = customer_months.pivot_table(
    index="acquisition_month", columns="period_number", values="customer_id", aggfunc="nunique", fill_value=0
)
retention = cohort_counts.div(cohort_counts[0], axis=0)
retention

### Exercise 48: Capstone G — investigate return risk

Compare return rates by category and discount band. Require at least 20 items per cell, and sort highest risk first.

<details><summary>Hint</summary>Bin discounts, aggregate item count and mean of the boolean return flag, then filter reliable cells.</details>

**Worked solution:**

In [ ]:
return_risk = (
    capstone_items.assign(
        discount_band=pd.cut(
            capstone_items["discount_pct"],
            bins=[-0.001, 0, 0.05, 0.10, 1],
            labels=["none", "light", "medium", "heavy"],
        )
    )
    .groupby(["category", "discount_band"], observed=True, as_index=False)
    .agg(items=("item_id", "size"), return_rate=("was_returned", "mean"))
    .loc[lambda d: d["items"].ge(20)]
    .sort_values("return_rate", ascending=False)
)
return_risk

### Exercise 49: Capstone H — write the decision memo

Write 4–6 bullets in a Markdown cell: overall performance, strongest/weakest category, trend, customer insight, return-risk insight, and one action. Every claim must cite a number from your tables.

**Worked solution:**

In [ ]:
display(executive_kpis)
display(monthly_kpis)
display(category_scorecard)
display(return_risk.head())
# The memo is intentionally open-ended. A strong answer cites calculated values,
# distinguishes facts from interpretations, and recommends one measurable action.

## 11. Final recall drill — do this without running code

Write the pandas expression you would reach for first:

1. Select rows by condition and two named columns.
2. Add a group total back to every original row.
3. Attach one customer record to many orders safely.
4. Find left-table records with no right-table match.
5. Convert a wide table to tidy long form.
6. Get the prior timestamp within each customer.
7. Calculate daily totals from timestamps.
8. Rank products within category.
9. Detect whether a supposed key is duplicated.
10. Conditionally update a column without chained assignment.

<details><summary>Compact recall key</summary>

1. `.loc[mask, [col1, col2]]`
2. `.groupby(key)[value].transform('sum')`
3. `.merge(..., validate='many_to_one')`
4. `.merge(..., indicator=True)` then filter `left_only`
5. `.melt(...)`
6. sort, then `.groupby(key)[time].shift()`
7. set a DatetimeIndex, then `.resample('D').sum()`
8. `.groupby(category)[value].rank(...)`
9. `.duplicated(key).any()`
10. `.loc[mask, col] = value`

</details>

## Pandas decision cheat sheet

| Need | Reach for |
|---|---|
| Inspect shape/types/missingness | `.shape`, `.info()`, `.isna().sum()`, custom audit |
| Pick rows/columns | `.loc`, `.iloc`, `query` |
| Create columns | `.assign`, vectorized operators, `np.select`, `cut` |
| Clean text | `.str` methods |
| Work with dates | `pd.to_datetime`, `.dt`, `resample` |
| Reduce each group | `groupby().agg()` |
| Add group facts to rows | `groupby().transform()` |
| Keep/remove whole groups | `groupby().filter()` |
| Combine columns from tables | `merge(..., validate=...)` |
| Stack row partitions | `concat` |
| Wide ↔ long | `pivot_table`, `melt`, `stack`, `unstack` |
| Previous/moving/cumulative | `shift`, `rolling`, `expanding`, `cumsum` |
| Top-N | `nlargest`, `nsmallest`, grouped `rank` |
| Validate assumptions | `assert`, `.duplicated`, row counts, merge indicator |

### The five questions to ask before every analysis

1. What does one row represent right now?
2. What should one row represent after this operation?
3. Which columns form the key?
4. Can this operation duplicate or drop rows?
5. Are my dtypes and missing values compatible with the question?

### Tomorrow's 25-minute retention loop

- Rebuild `line_items` from memory.
- Re-solve one GroupBy, one merge, one reshape, and one window exercise.
- Explain `agg` versus `transform` aloud.
- Recreate the executive KPI card without looking.

If you can complete the capstone cleanly and explain your grain at each step, you have closed the practical pandas gap.